In [ ]:
import fastf1
import fastf1.plotting
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import torch

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")
CACHE_DIR = '../cache'
Path(CACHE_DIR).mkdir(exist_ok=True)
fastf1.Cache.enable_cache(CACHE_DIR)

plt.style.use('dark_background')  # optional, looks nice for F1 data
%matplotlib inline

In [ ]:
session = fastf1.get_session(2023, 1, 'R')  # Bahrain
session.load(telemetry=False, weather=True, messages=False)

laps = session.laps.pick_quicklaps()
print(f"Total laps: {len(laps)}")
print(f"\nColumns available:\n{laps.columns.tolist()}")
print(f"\nCompounds used: {laps['Compound'].unique()}")
print(f"Drivers: {laps['Driver'].unique()}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
laps['LapTimeSeconds'] = laps['LapTime'].dt.total_seconds()

ax.hist(laps['LapTimeSeconds'], bins=40, color='steelblue', edgecolor='white')
ax.set_xlabel('Lap time (s)')
ax.set_ylabel('Count')
ax.set_title('Lap time distribution — 2023 Bahrain')
plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

for compound, group in laps.groupby('Compound'):
    group_sorted = group.sort_values('TyreLife')
    ax.scatter(group_sorted['TyreLife'], group_sorted['LapTimeSeconds'],
               label=compound, alpha=0.5, s=15)

ax.set_xlabel('Tyre life (laps)')
ax.set_ylabel('Lap time (s)')
ax.set_title('Tire degradation by compound — 2023 Bahrain')
ax.legend()
plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

for ax, (compound, group) in zip(axes, laps.groupby('Compound')):
    group_sorted = group.sort_values('TyreLife')
    rolling = group_sorted.groupby('TyreLife')['LapTimeSeconds'].median()
    ax.plot(rolling.index, rolling.values, marker='o', markersize=3)
    ax.set_title(compound)
    ax.set_xlabel('Tyre life (laps)')
    ax.set_ylabel('Median lap time (s)')

plt.suptitle('Degradation trend per compound', y=1.02)
plt.tight_layout()

In [ ]:
top_drivers = laps['Driver'].value_counts().head(5).index
fig, ax = plt.subplots(figsize=(12, 5))

for driver in top_drivers:
    driver_laps = laps[laps['Driver'] == driver].sort_values('LapNumber')
    ax.plot(driver_laps['LapNumber'], driver_laps['LapTimeSeconds'],
            label=driver, alpha=0.7, linewidth=1.2)

ax.set_xlabel('Lap number')
ax.set_ylabel('Lap time (s)')
ax.set_title('Lap time evolution per driver — 2023 Bahrain')
ax.legend()
plt.tight_layout()

In [ ]:
weather = session.weather_data.sort_values('Time')
laps_merged = pd.merge_asof(
    laps.sort_values('LapStartTime'),
    weather[['Time', 'AirTemp', 'TrackTemp']],
    left_on='LapStartTime',
    right_on='Time',
    direction='nearest'
)

laps_merged['FuelLoad'] = 110 - (laps_merged['LapNumber'] * 1.6)
laps_merged['StintLength'] = laps_merged['TyreLife']

numeric_cols = ['LapTimeSeconds', 'StintLength', 'FuelLoad', 'AirTemp', 'TrackTemp']
corr = laps_merged[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=ax, square=True)
ax.set_title('Feature correlation — 2023 Bahrain')
plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

drivers = laps.sort_values('LapNumber')['Driver'].unique()
colors  = {'SOFT': '#e8473f', 'MEDIUM': '#f0d05a', 'HARD': '#e8e8e8'}

for i, driver in enumerate(drivers):
    driver_laps = laps[laps['Driver'] == driver].sort_values('LapNumber')
    for _, lap in driver_laps.iterrows():
        color = colors.get(lap['Compound'], 'gray')
        ax.barh(i, 1, left=lap['LapNumber'], color=color,
                edgecolor='none', height=0.6)

ax.set_yticks(range(len(drivers)))
ax.set_yticklabels(drivers, fontsize=8)
ax.set_xlabel('Lap number')
ax.set_title('Stint structure by driver — 2023 Bahrain')

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=k) for k, c in colors.items()]
ax.legend(handles=legend_elements, loc='lower right')
plt.tight_layout()